# IAP Spike Features → LFP Features: Ridge Regression (Population)

**Question:** Do intracellular spike waveform features predict the state of the LFP in the ~50 ms around each spike, and does this hold consistently across cells?

**Data model:**
- Spike features loaded from `cluster_pickles/cXX_cluster_df.pkl` (same QC'd features used in all cluster analyses)
- LFP features extracted by averaging sliding-window values over a 50 ms pre/post window:
  - LFP amp, LFP std — from `simple_lfp_pickles/cXX_simple_lfp.pkl`
  - Aperiodic exponent, θ AUC, γ AUC — from `multitaper_pickles/cXX/` specparam chunks

**Two-level stats:**
1. **Per cell** — RidgeCV betas + permutation test (1000×) + FDR correction within cell
2. **Population** — one-sample t-test on betas across cells + FDR correction across all tests

In [ ]:
# ── Run-control flags ─────────────────────────────────────────────────────────
FORCE_RERUN = False   # set True to recompute and overwrite pickles


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
import sys, os, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore', message='use_inf_as_na option is deprecated',
                        category=FutureWarning, module='seaborn')
warnings.filterwarnings('ignore', message=r'invalid value encountered in log10',
                        category=RuntimeWarning, module=r'.*specparam.*')

# paths
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikewise/spikewise')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikewise/Action potential waveforms are state-dependent paper/AP_empirical_paper_all_analyses/datasets/spe-1/spe1_helper_modules')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikewise/Action potential waveforms are state-dependent paper/AP_empirical_paper_all_analyses/datasets/shared_helper_modules')

# spe-1 helpers (same imports used in all per-cell ridge notebooks)
from spk_feat_cluster_analysis import load_chunked_specparam_results, trim_edges
from config import CELL_IDS, SPE1_PICKLE_ROOT

# our utils
from spe1_ridge_utils import (
    load_cell_data,
    build_cell_df,
    run_ridge_cell,
    run_population_analysis,
    plot_cell_betas,
    plot_population_betas,
    plot_beta_heatmap,
    plot_significant_scatters,
    plot_cell_summary_scatter,
    SPIKE_FEATURES, SPIKE_FEATURES_WITH_ISI,
    LFP_TARGETS, PRE_WIN, POST_WIN, MIN_SPIKES, N_PERMS,
)

print('imports ok')

---
## 1. Config

In [ ]:
# Which cells to include — all cells from config, or override with a subset
CELLS_TO_RUN = [int(c.replace('c', '')) for c in CELL_IDS]   # all cells
# CELLS_TO_RUN = [1, 3, 4, 14, 21, 24, 26, 42]               # subset example

# Analysis options
INCLUDE_LOG_ISI = True
SPIKE_FEATS     = SPIKE_FEATURES_WITH_ISI if INCLUDE_LOG_ISI else SPIKE_FEATURES
N_PERMS_RUN     = N_PERMS   # 1000 — lower to e.g. 100 for quick test runs

# Time windows for LFP feature extraction (seconds, relative to spike at t=0)
# Both windows are 50 ms, skipping the +/-5 ms immediately around the spike.
PRE_WIN_RUN  = PRE_WIN    # (-0.055, -0.005)
POST_WIN_RUN = POST_WIN   # ( 0.005,  0.055)

print(f'Spike features  : {SPIKE_FEATS}')
print(f'LFP targets     : {LFP_TARGETS}')
print(f'Pre window  (s) : {PRE_WIN_RUN}')
print(f'Post window (s) : {POST_WIN_RUN}')
print(f'Cells to run    : {CELLS_TO_RUN}')
print(f'Permutations    : {N_PERMS_RUN}')

---
## 2. Single-cell demo (sanity check)

Load one cell from pickles, build the feature DataFrame, and inspect the
raw IAP → LFP scatter before running the permutation test.

In [ ]:
demo_num = 14

cluster_df_demo, lfp_win_demo, sp_demo, n_demo = load_cell_data(
    cell_num                       = demo_num,
    pickle_root                    = SPE1_PICKLE_ROOT,
    load_chunked_specparam_results = load_chunked_specparam_results,
    trim_edges                     = trim_edges,
)

print(f'c{demo_num}  ->  {n_demo} spikes')
print(f'  cluster_df columns : {list(cluster_df_demo.columns)}')
print(f'  simple_lfp keys    : {list(lfp_win_demo[0].keys()) if lfp_win_demo[0] else "None"}')
print(f'  specparam keys     : {list(sp_demo[0].keys()) if sp_demo[0] else "None"}')

In [ ]:
df_demo = build_cell_df(
    cluster_df           = cluster_df_demo,
    lfp_windows_by_spike = lfp_win_demo,
    specparam_by_spike   = sp_demo,
    pre_win              = PRE_WIN_RUN,
    post_win             = POST_WIN_RUN,
    include_log_isi      = INCLUDE_LOG_ISI,
)

print(f'Shape after NaN drop: {df_demo.shape}   (spikes x features)')
df_demo.describe()

In [ ]:
# Sanity check: scatter IAP features vs LFP targets (Pearson r, uncorrected)
plot_cell_summary_scatter(df_demo, spike_features=SPIKE_FEATS)

---
## 3. Per-cell ridge regression + permutation test (demo cell)

- X and y **z-scored within cell** -> beta weights comparable across features and cells
- **RidgeCV** picks best L2 penalty via 5-fold CV
- **1000 permutations** of z-scored y -> null beta distribution
- Two-tailed p = fraction of |null beta| >= |real beta| per feature
- **FDR** (Benjamini-Hochberg) across all feature x target combos within cell

In [ ]:
betas_demo, pvals_demo, alphas_demo = run_ridge_cell(
    df_cell        = df_demo,
    spike_features = SPIKE_FEATS,
    lfp_targets    = LFP_TARGETS,
    n_perms        = N_PERMS_RUN,
    show_progress  = True,
)

print('\nBest alpha per LFP target:')
for t, a in alphas_demo.items():
    print(f'  {t:<22} alpha = {a:.4f}')

In [ ]:
# bar chart: red = FDR-significant, blue spine = pre, orange spine = post
plot_cell_betas(betas_demo, pvals_demo, cell_id=f'c{demo_num}', spike_features=SPIKE_FEATS)

---
## 4. Full run — all cells

In [ ]:
all_betas   = []
all_pvals   = []
cell_ids_ok = []
cell_dfs    = {}

for cell_num in tqdm(CELLS_TO_RUN, desc='Cells'):
    cell_id = f'c{cell_num}'
    print(f'\n-- {cell_id} --------------------------------------------------')

    try:
        _cluster, _lfp_win, _sp, _n = load_cell_data(
            cell_num                       = cell_num,
            pickle_root                    = SPE1_PICKLE_ROOT,
            load_chunked_specparam_results = load_chunked_specparam_results,
            trim_edges                     = trim_edges,
        )
    except FileNotFoundError as e:
        print(f'  SKIP -- pickle not found: {e}')
        continue

    print(f'  Spikes loaded: {_n}')

    _df = build_cell_df(
        cluster_df           = _cluster,
        lfp_windows_by_spike = _lfp_win,
        specparam_by_spike   = _sp,
        pre_win              = PRE_WIN_RUN,
        post_win             = POST_WIN_RUN,
        include_log_isi      = INCLUDE_LOG_ISI,
    )

    if len(_df) < MIN_SPIKES:
        print(f'  SKIP -- only {len(_df)} clean spikes (min={MIN_SPIKES})')
        continue
    print(f'  Clean spikes for regression: {len(_df)}')

    _betas, _pvals, _alphas = run_ridge_cell(
        df_cell        = _df,
        spike_features = SPIKE_FEATS,
        lfp_targets    = LFP_TARGETS,
        n_perms        = N_PERMS_RUN,
        show_progress  = False,
    )

    cell_dfs[cell_id]   = _df
    all_betas.append(_betas)
    all_pvals.append(_pvals)
    cell_ids_ok.append(cell_id)
    print(f'  Done.  alpha range: {min(_alphas.values()):.3f} - {max(_alphas.values()):.3f}')

print(f'\nCompleted: {len(cell_ids_ok)} / {len(CELLS_TO_RUN)} cells')

In [ ]:
import os
RESULTS_DIR = os.path.join(SPE1_PICKLE_ROOT, '..', 'ridge_results')
os.makedirs(RESULTS_DIR, exist_ok=True)

results = {
    'all_betas'      : all_betas,
    'all_pvals'      : all_pvals,
    'cell_ids'       : cell_ids_ok,
    'spike_feats'    : SPIKE_FEATS,
    'lfp_targets'    : LFP_TARGETS,
    'pre_win'        : PRE_WIN_RUN,
    'post_win'       : POST_WIN_RUN,
    'n_perms'        : N_PERMS_RUN,
    'include_log_isi': INCLUDE_LOG_ISI,
}
save_path = os.path.join(RESULTS_DIR, 'ridge_results.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(results, f)
print(f'Saved -> {save_path}')

In [ ]:
# (optional) reload without re-running
# with open(save_path, 'rb') as f:
#     results = pickle.load(f)
# all_betas   = results['all_betas']
# all_pvals   = results['all_pvals']
# cell_ids_ok = results['cell_ids']
# SPIKE_FEATS = results['spike_feats']
# print(f'Loaded results for {len(cell_ids_ok)} cells')

---
## 5. Per-cell beta plots

In [ ]:
for cell_id, betas, pvals in zip(cell_ids_ok, all_betas, all_pvals):
    plot_cell_betas(betas, pvals, cell_id=cell_id, spike_features=SPIKE_FEATS)

---
## 6. Population analysis

**One-sample t-test** on the beta distribution across cells (H0: mean beta = 0),
then **FDR correction** across all feature x target tests.

In [ ]:
mean_betas, sem_betas, tstats, pvals_fdr, n_cells_df = run_population_analysis(
    all_betas      = all_betas,
    lfp_targets    = LFP_TARGETS,
    spike_features = SPIKE_FEATS,
)

print(f'N cells: {len(cell_ids_ok)}\n')
print('=== Significant pairs (FDR p<0.05) ===')
sig_rows = []
for target in LFP_TARGETS:
    for feat in SPIKE_FEATS:
        p = pvals_fdr.loc[target, feat]
        if not np.isnan(p) and p < 0.05:
            sig_rows.append({
                'target'   : target,
                'feature'  : feat,
                'mean_beta': round(mean_betas.loc[target, feat], 3),
                't'        : round(tstats.loc[target, feat],     3),
                'p_fdr'    : round(p, 4),
            })
if sig_rows:
    print(pd.DataFrame(sig_rows).to_string(index=False))
else:
    print('  None at FDR p<0.05')

---
## 7. Population visualizations

In [ ]:
# Raincloud: violin + individual cell dots + mean +/- SEM + population FDR stars
plot_population_betas(
    all_betas      = all_betas,
    pvals_fdr      = pvals_fdr,
    lfp_targets    = LFP_TARGETS,
    spike_features = SPIKE_FEATS,
)

In [ ]:
# Heatmap: mean beta with significance stars
plot_beta_heatmap(
    mean_betas     = mean_betas,
    pvals_fdr      = pvals_fdr,
    spike_features = SPIKE_FEATS,
)

---
## 8. Significant findings — scatter plots

For every feature x LFP target pair that survives population FDR correction,
plot the raw relationship: spikes pooled across cells (z-scored within cell),
with per-cell regression lines and pooled Pearson r annotated.

In [ ]:
plot_significant_scatters(
    cell_dfs       = cell_dfs,
    cell_ids       = cell_ids_ok,
    pvals_fdr      = pvals_fdr,
    mean_betas     = mean_betas,
    spike_features = SPIKE_FEATS,
    lfp_targets    = LFP_TARGETS,
)

---
## 9. Control: spike shape features without log ISI

Re-run without log ISI in X. If spike shape betas hold up, the waveform -> LFP
relationship is not just a firing rate effect.

In [ ]:
all_betas_noisi = []

for cell_id, df_cell in tqdm(cell_dfs.items(), desc='ISI control'):
    _b, _p, _ = run_ridge_cell(
        df_cell        = df_cell,
        spike_features = SPIKE_FEATURES,   # no log ISI
        lfp_targets    = LFP_TARGETS,
        n_perms        = N_PERMS_RUN,
        show_progress  = False,
    )
    all_betas_noisi.append(_b)

mean_b_noisi, _, _, pvals_fdr_noisi, _ = run_population_analysis(
    all_betas      = all_betas_noisi,
    lfp_targets    = LFP_TARGETS,
    spike_features = SPIKE_FEATURES,
)

plot_beta_heatmap(
    mean_betas     = mean_b_noisi,
    pvals_fdr      = pvals_fdr_noisi,
    spike_features = SPIKE_FEATURES,
    title          = 'Mean beta weights -- spike shape only (no log ISI)',
)

plot_significant_scatters(
    cell_dfs       = cell_dfs,
    cell_ids       = cell_ids_ok,
    pvals_fdr      = pvals_fdr_noisi,
    mean_betas     = mean_b_noisi,
    spike_features = SPIKE_FEATURES,
    lfp_targets    = LFP_TARGETS,
)